## <b><font color='darkblue'>Preface</font></b>
([source](https://adk.dev/skills/)) <font size='3ptx'><b>An agent `Skill` is a self-contained unit of functionality that an ADK agent can use to perform a specific task</b>. An agent Skill encapsulates the necessary instructions, resources, and tools required for a task, based on the [**Agent Skill specification**](https://agentskills.io/specification). The structure of a Skill allows it to be loaded incrementally to minimize the impact on the operating context window of the agent.</font>

In [17]:
import agent_utils
import os
import pathlib

from google.adk import Agent
from google.adk.tools.base_tool import BaseTool
from google.adk.skills import load_skill_from_dir
from google.adk.tools import skill_toolset
from google.genai import types


if 'GEMINI_API_KEY' in os.environ:
    del os.environ['GEMINI_API_KEY']
    
AgentBroker = agent_utils.AgentBroker
AGENT_ROOT = pathlib.Path(os.getcwd()).cwd()

## <b><font color='darkblue'>Get started</font></b>
<b><font size='3ptx'>Use the `SkillToolset` class to make one or more Skills available to your agent.</font></b>

You can define [skills in code](https://adk.dev/skills/#inline-skills) or [load skills from a filesystem](https://adk.dev/skills/#filesystem-skills). Below is the file structure of skills:
```shell
$ tree skills/
skills/
└── weather-skill
    ├── references
    │   └── weather_info.md
    ├── scripts
    │   └── get_humidity.py
    └── SKILL.md
```

In [2]:
pathlib.Path(os.getcwd()).cwd()

PosixPath('/usr/local/google/home/johnkclee/Github/ml_articles/google/agent_development_kit')

In [3]:
weather_skill = load_skill_from_dir(
    pathlib.Path(os.getcwd()).cwd() / "skills" / "weather-skill"
)

In [4]:
class GetTimezoneTool(BaseTool):
  """A tool to get the timezone for a given location."""

  def __init__(self):
    super().__init__(
        name="get_timezone",
        description="Returns the timezone for a given location.",
    )

  def _get_declaration(self) -> types.FunctionDeclaration | None:
    return types.FunctionDeclaration(
        name=self.name,
        description=self.description,
        parameters_json_schema={
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The location to get the timezone for.",
                },
            },
            "required": ["location"],
        },
    )

In [5]:
def get_wind_speed(location: str) -> str:
  """Returns the current wind speed for a given location."""
  return f"The wind speed in {location} is 10 mph."

    
my_skill_toolset = skill_toolset.SkillToolset(
    skills=[weather_skill],
    additional_tools=[GetTimezoneTool(), get_wind_speed],
)

/usr/local/google/home/johnkclee/Github/ml_articles/env/lib/python3.13/site-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.SKILL_TOOLSET is enabled.
  check_feature_enabled()


In [6]:
root_agent = Agent(
    model="gemini-flash-latest",
    name="skill_user_agent",
    description="An agent that can use specialized skills.",
    instruction=(
        "You are a helpful assistant that can leverage skills to perform tasks."
    ),
    tools=[
        my_skill_toolset,
    ],
)

For a complete code example of an ADK agent with a Skill, including both file-based and in-line Skill definitions, see the code sample [skills_agent](https://github.com/google/adk-python/blob/main/contributing/samples/skills_agent/agent.py).

Now, let's test this agent a bit:

In [11]:
agent_broker = AgentBroker(root_agent)

In [12]:
resp = await agent_broker.interact("what's weather in Taiwan?")

In [13]:
print(resp.text)

The current weather in Taipei, Taiwan is sunny with a temperature of 72°F (22°C). The forecast predicts clear skies all day.


## <b><font color='darkblue'>Understand Skills</font></b>
<font size='3ptx'><b>The Skills feature allows you to create modular packages of Skill instructions and resources that agents can load on demand.</b> This approach helps you organize your agent's capabilities and optimize the context window by only loading instructions when they are needed.</font>

The structure of Skills is organized into three levels:
- **L1 (`Metadata`)**: Provides metadata for skill discovery. This information is defined in the frontmatter section of the <font color='olive'>SKILL.md</font> file and includes properties such as the Skill name and description.
- **L2 (`Instructions`)**: Contains the primary instructions for the Skill, loaded when the Skill is triggered by the agent. This information is defined in the body of the <font color='olive'>SKILL.md</font> file.
- **L3 (`Resources`)**: Includes additional resources such as reference materials, assets, and scripts that can be loaded as needed. These resources are organized into the following directories:
    - **`references/`**: Additional Markdown files with extended instructions, workflows, or guidance.
    - **`assets/`**: Resource materials such as database schemas, API documentation, templates, or examples.
    - **`scripts/`**: Executable scripts supported by the agent runtime.

![Difference between `references` and `assets`](images/references_vs_assets.png)

### <b><font color='darkgreen'>Skills directory structure</font></b>
<font size='3ptx'><b>The following directory structure shows the recommended way to include Skills in your ADK agent project.</font></b>

The <font color='olive'>example-skill/</font> directory shown below, and any parallel Skill directories, must follow the [**Agent Skill specification**](https://agentskills.io/specification) file structure. Only the <font color='olive'>SKILL.md</font> file is required:
```
my_agent/
    agent.py (or agent.ts / main.go)
    .env
    skills/
        example-skill/        # Skill
            SKILL.md          # main instructions (required)
            references/
                REFERENCE.md  # detailed API reference
                FORMS.md      # form-filling guide
                *.md          # domain-specific information
            assets/
                *.*           # templates, images, data
            scripts/
                *.py          # utility scripts (Python)
                *.js          # utility scripts (JavaScript)
                *.ts          # utility scripts (TypeScript)
```

## <b><font color='darkblue'>Skill sources</font></b>
You can define [skills within the code](https://adk.dev/skills/#inline-skills) or [read skills from a filesystem](https://adk.dev/skills/#filesystem-skills).

### <b><font color='darkgreen'>Define Skills in code</font></b>
You can define Skills within the code of your agent, as shown below.

In [14]:
from google.adk.skills import models

greeting_skill = models.Skill(
    frontmatter=models.Frontmatter(
        name="greeting-skill",
        description=(
            "A friendly greeting skill that can say hello to a specific person."
        ),
    ),
    instructions=(
        "Step 1: Read the 'references/hello_world.txt' file to understand how"
        " to greet the user. Step 2: Return a greeting based on the reference."
    ),
    resources=models.Resources(
        references={
            "hello_world.txt": "Hello! So glad to have you here!",
            "example.md": "This is an example reference.",
        },
    ),
)

Here is the breakdown of the three primary components:
1. **frontmatter** (`Identity`): The <font color='blue'>**Frontmatter**</font> object handles the metadata.
    - `name`: This is the unique identifier for the skill (<font color='brown'>"greeting-skill"</font>). This is how the ADK registry and other skills will reference it.
    - `description`: A plain-text explanation of the skill's purpose. This is crucial because, in larger systems, a "Router" or "Orchestrator" agent reads this description to decide if this specific skill is the right tool to handle a user's request.
2. **instructions** (`Logic`): This is a string of natural language prompts that dictate the agent's behavior.
    - Unlike traditional programming where you write `if/else` statements, here you provide declarative steps.
    - Step 1 tells the agent to look at a specific resource (<font color='brown'>the "reference" file</font>).
    - Step 2 defines the final output goal.
    - The agent uses these instructions to "reason" through the task.
4. **resources** (`Knowledge Base`): The Resources object provides the "context" or "memory" for the skill.
    - `references`: This is a dictionary acting as a virtual file system.
    - In your snippet, `"hello_world.txt"` isn't a real file on your hard drive; it is a key that maps to the string "Hello! So glad to have you here!".
    - When the instructions tell the agent to "Read the <font color='olive'>'references/hello_world.txt'</font> file," the agent looks into this dictionary, retrieves the string, and uses it to fulfill the request.

How it works in practice? When a user says "Hi there!", the ADK processes this skill by:
1. Identifying that the <font color='blue'>**Frontmatter**</font> matches the user's intent.
2. Following the <b>Instructions</b> sequentially.
3. <b>Injecting the text from Resources into the LLM's prompt window</b> so it knows exactly how you want the greeting formatted (<font color='brown'>e.g., "So glad to have you here!"</font>).

<b><font color='orange'>Key Takeaway</font></b>: This structure separates the identity (frontmatter), the logic (instructions), and the data (resources), making the skill modular and easy to update without changing the underlying code structure.

### <b><font color='darkgreen'>Read Skills from filesystem</font></b>
You can load skills from file system with file structure too. Below is the file system structure of skill `weather-skill` for example:
```shell
$ tree skills/
skills/
└── weather-skill
    ├── references
    │   └── weather_info.md
    ├── scripts
    │   └── get_humidity.py
    └── SKILL.md
```

In [21]:
import pathlib

from google.adk.skills import load_skill_from_dir
from google.adk.tools import skill_toolset


weather_skill = load_skill_from_dir(
    pathlib.Path(AGENT_ROOT).cwd() / "skills" / "weather-skill"
)

my_skill_toolset = skill_toolset.SkillToolset(
    skills=[weather_skill],
)